# IOT_Based_Crop_Disease_Detection_System_For_Smart_Agriculture


### Import packages

In [2]:
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import torch
import torchvision
import torch.nn as nn
from torchvision import models
import torch.optim as optim


from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

### Check GPU / CPU

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Import Datast

### Mount Google Drive

In [4]:
from google.colab import drive # type: ignore
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load Train , Valid , Test 

In [5]:
from torchvision import datasets, transforms

In [6]:
# Define preprocessing and augmentation

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.9, 1.0)
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = valid_transform

In [7]:
import os

base_path = "/content/drive/MyDrive/dataset/Crop_Disease_Split"

print(os.listdir(base_path))

['test', 'train', 'valid']


In [8]:
train_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/valid",
    transform=valid_transform
)

test_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/dataset/Crop_Disease_Split/test",
    transform=test_transform
)

### train_transform 
- The training transform takes a raw JPG/JPEG RGB image, applies random data augmentation (crop, flip, and rotation), converts it into a normalized float32 PyTorch tensor of shape (3, 224, 224), and prepares it as input for EfficientNet-B0 during model training.

In [18]:
from torch.utils.data import DataLoader

# Train DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Validation DataLoader
valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=" * 50)
print("Train Images      :", len(train_dataset))
print("Validation Images :", len(valid_dataset))
print("Test Images       :", len(test_dataset))
print("=" * 50)
print("Train Batches     :", len(train_loader))
print("Validation Batches:", len(valid_loader))
print("Test Batches      :", len(test_loader))
print("=" * 50)

Train Images      : 15512
Validation Images : 3325
Test Images       : 3332
Train Batches     : 485
Validation Batches: 104
Test Batches      : 105


### Load the pretrained EfficientNet-B0 model

In [10]:

# Number of classes
NUM_CLASSES = 19

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Replace the final classifier
model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

# Select device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model = model.to(device)

# Print model information
print(model)
print("\nOutput classes:", model.classifier[1].out_features)
print("Device:", device)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [11]:
# Loss Function

criterion = nn.CrossEntropyLoss()

In [13]:
# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [14]:
# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3
)

In [19]:
from tqdm.auto import tqdm
import time

# Number of training epochs
num_epochs = 20

for epoch in range(num_epochs):

    start_time = time.time()

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        unit="batch"
    )

    for batch_idx, (images, labels) in enumerate(progress_bar):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        current_loss = running_loss / (batch_idx + 1)
        current_accuracy = 100 * correct / total

        progress_bar.set_postfix(
            Loss=f"{current_loss:.4f}",
            Accuracy=f"{current_accuracy:.2f}%"
        )

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    scheduler.step(epoch_loss)

    epoch_time = time.time() - start_time

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{num_epochs} Completed")
    print(f"Training Loss     : {epoch_loss:.4f}")
    print(f"Training Accuracy : {epoch_accuracy:.2f}%")
    print(f"Epoch Time        : {epoch_time:.2f} sec")
    print("=" * 60)

Epoch 1/20:   0%|          | 0/485 [00:00<?, ?batch/s]


Epoch 1/20 Completed
Training Loss     : 0.3668
Training Accuracy : 87.36%
Epoch Time        : 1121.08 sec


Epoch 2/20:   0%|          | 0/485 [00:00<?, ?batch/s]


Epoch 2/20 Completed
Training Loss     : 0.2239
Training Accuracy : 92.34%
Epoch Time        : 198.57 sec


Epoch 3/20:   0%|          | 0/485 [00:00<?, ?batch/s]


Epoch 3/20 Completed
Training Loss     : 0.1939
Training Accuracy : 93.39%
Epoch Time        : 192.87 sec


Epoch 4/20:   0%|          | 0/485 [00:00<?, ?batch/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ddb1d6d84a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
 Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7ddb1d6d84a0> 
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
       
self._shutdown_workers()    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
 ^    ^if w.is_alive():^
^ ^ ^ ^ ^ ^ ^ ^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
# Evaluate on Test Dataset

model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss /= len(test_loader)
test_accuracy = 100 * correct / total

print("=" * 50)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.2f}%")
print("=" * 50)

In [ ]:
# classification report 

from sklearn.metrics import classification_report

model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=train_dataset.classes
    )
)

In [ ]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(14,12))

plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = range(len(train_dataset.classes))

plt.xticks(tick_marks, train_dataset.classes, rotation=90)
plt.yticks(tick_marks, train_dataset.classes)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()
plt.show()

In [ ]:
import os

# Create folder if it doesn't exist
save_dir = "/content/drive/MyDrive/Trained_Models"
os.makedirs(save_dir, exist_ok=True)

# Model save path
model_path = os.path.join(save_dir, "efficientnet_b0_crop_disease.pth")

# Save model
torch.save(model.state_dict(), model_path)

print("=" * 50)
print("Model saved successfully!")
print(f"Saved at: {model_path}")
print("=" * 50)

In [ ]:
# Save Class Names
import os
import json

# Create folder if it doesn't exist
save_dir = "/content/drive/MyDrive/Trained_Models"
os.makedirs(save_dir, exist_ok=True)

# Save class names
class_path = os.path.join(save_dir, "class_names.json")

with open(class_path, "w") as f:
    json.dump(train_dataset.classes, f, indent=4)

print("=" * 50)
print("Class names saved successfully!")
print(f"Saved at: {class_path}")
print("=" * 50)

In [ ]:
# Save the class names to Google Drive

import json
import os

class_path = os.path.join(save_dir, "class_names.json")

with open(class_path, "w") as f:
    json.dump(train_dataset.classes, f, indent=4)

print("=" * 50)
print("Class names saved successfully!")
print(f"Location: {class_path}")
print("=" * 50)